# SGLang 适配国产开源大模型 — 实操教程

**定位：** 演示如何使用 SGLang 部署和调用国产开源大语言模型，覆盖 Qwen、ChatGLM、DeepSeek 等主流中文模型的适配要点和实操流程。

```bash
# 默认启动命令（以 Qwen2.5-0.5B 为例）
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA GPU（算力 ≥ 7.0） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB（0.5B 模型） | ≥ 16 GB（6B 模型） |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 安装 SGLang 全部组件及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 Python 环境正常
2. 如需安装依赖，运行「可选安装」单元格
3. 阅读国产大模型生态介绍和模型选择参考表
4. 运行「通用启动函数」单元格定义服务启动工具
5. 依次运行各模型的 Demo 单元格体验不同模型
6. 运行「中文推理效果测试」对比不同任务表现
7. 完成后运行「清理资源」单元格释放 GPU 显存

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

print('=' * 50)  # 打印分隔线
print('SGLang 国产大模型适配 — 环境检查')  # 打印标题
print('=' * 50)  # 打印分隔线

print(f'Python 版本: {sys.version}')  # 输出当前 Python 版本

try:  # 尝试执行 nvidia-smi 命令
    result = subprocess.run(  # 运行 nvidia-smi 查询 GPU 信息
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出并设置超时
    )  # 命令执行完成
    if result.returncode == 0:  # 如果命令执行成功
        print(f'GPU 信息: {result.stdout.strip()}')  # 输出 GPU 信息
    else:  # 如果命令执行失败
        print('警告: nvidia-smi 执行失败')  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print('警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动')  # 提示用户

try:  # 尝试导入 sglang
    import sglang  # 导入 sglang 模块
    print(f'SGLang 版本: {sglang.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('SGLang 未安装，请运行下方安装单元格')  # 提示安装

try:  # 尝试导入 openai
    import openai  # 导入 openai 模块
    print(f'OpenAI SDK 版本: {openai.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('OpenAI SDK 未安装')  # 提示安装

print('=' * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# import subprocess  # 导入子进程模块
# import sys  # 导入系统模块
# subprocess.check_call([  # 执行 pip 安装命令
#     sys.executable, '-m', 'pip', 'install', '-q',  # 使用当前解释器静默安装
#     'sglang[all]', 'openai>=1.0.0', 'requests>=2.28.0'  # 安装所需依赖
# ])  # 安装命令执行完成
# print('依赖安装完成')  # 打印安装成功提示

## 第一部分：国产大模型生态

近年来，中国在大语言模型领域取得了显著进展，涌现出众多优秀的开源模型：

- **Qwen 系列（通义千问）：** 由阿里云推出，从 0.5B 到 72B 参数量全覆盖，中文理解和生成能力优秀，社区活跃
- **ChatGLM 系列：** 由清华大学 THUDM 团队开发，中英双语对话模型，ChatGLM3-6B 支持工具调用和代码执行
- **DeepSeek 系列：** 由深度求索推出，DeepSeek-R1 系列推理能力尤其突出，Distill 版本适合资源受限场景
- **Baichuan 系列：** 百川智能推出的开源大模型，专注中文场景优化
- **Yi 系列：** 零一万物推出的开源模型，多语言能力强

SGLang 对主流国产开源大模型有良好的支持，本 Notebook 将演示如何部署和使用这些模型。

## 第二部分：模型选择参考表

| 模型名称 | HuggingFace ID | 参数量 | 最低显存 | 特点 |
|---------|---------------|--------|---------|------|
| Qwen2.5-0.5B | `Qwen/Qwen2.5-0.5B-Instruct` | 0.5B | ≥ 6GB | 轻量级，适合入门测试和低资源环境 |
| Qwen2.5-1.5B | `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | ≥ 8GB | 能力更强，仍然轻量 |
| ChatGLM3-6B | `THUDM/chatglm3-6b` | 6B | ≥ 16GB | 需要 `--trust-remote-code`，中英双语 |
| DeepSeek-R1-Distill-Qwen-1.5B | `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | 1.5B | ≥ 8GB | 推理能力强，适合数学和逻辑任务 |

> **提示：** 实际显存需求还取决于 `--context-length`、`--mem-fraction-static` 等参数。首次使用时模型会自动从 HuggingFace 下载，请确保网络通畅。

In [ ]:
# === 启动服务：通用函数 ===
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求模块
import sys  # 导入系统模块

server_process = None  # 全局服务进程变量，初始为 None


def launch_model(  # 定义通用模型启动函数
    model_path,  # 模型路径（HuggingFace ID 或本地路径，必填）
    host='127.0.0.1',  # 服务监听地址（默认本机回环）
    port=30000,  # 服务端口号（默认 30000）
    trust_remote_code=False,  # 是否信任远程代码（默认关闭）
    dtype='auto',  # 数据类型（默认自动选择）
    extra_args=None,  # 额外启动参数列表（默认 None）
    max_wait=300,  # 最大等待时间秒数（默认 300）
):  # 函数参数定义完成
    """通用 SGLang 服务启动函数，支持任意 HuggingFace 模型"""  # 函数文档字符串
    global server_process  # 声明使用全局服务进程变量

    if server_process is not None:  # 如果已有运行中的服务进程
        print('正在终止旧服务进程...')  # 提示正在终止旧服务
        server_process.terminate()  # 发送终止信号给旧进程
        try:  # 尝试等待进程退出
            server_process.wait(timeout=15)  # 等待旧进程退出，最多 15 秒
        except subprocess.TimeoutExpired:  # 如果等待超时
            server_process.kill()  # 强制终止旧进程
        print('旧服务已终止')  # 打印终止成功信息
        time.sleep(3)  # 等待 3 秒确保端口释放

    cmd = [  # 构建启动命令列表
        sys.executable, '-m', 'sglang.launch_server',  # 调用 sglang 启动模块
        '--model-path', model_path,  # 指定模型路径
        '--host', host,  # 指定监听地址
        '--port', str(port),  # 指定端口号
        '--dtype', dtype,  # 指定数据类型
    ]  # 基础命令构建完成

    if trust_remote_code:  # 如果需要信任远程代码
        cmd.append('--trust-remote-code')  # 添加信任远程代码参数

    if extra_args:  # 如果有额外启动参数
        cmd.extend(extra_args)  # 将额外参数追加到命令中

    print(f'启动命令: {" ".join(cmd)}')  # 打印完整的启动命令

    server_process = subprocess.Popen(  # 以子进程方式启动 SGLang 服务
        cmd,  # 传入命令列表
        stdout=subprocess.PIPE,  # 捕获标准输出
        stderr=subprocess.PIPE,  # 捕获标准错误输出
    )  # 子进程创建完成
    print(f'服务进程已启动，PID = {server_process.pid}')  # 打印进程 ID

    start_time = time.time()  # 记录开始时间
    while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
        try:  # 尝试连接服务
            resp = requests.get(f'http://{host}:{port}/health', timeout=5)  # 发送健康检查请求
            if resp.status_code == 200:  # 如果服务返回 200 状态码
                elapsed = time.time() - start_time  # 计算启动耗时
                print(f'服务启动成功！耗时 {elapsed:.1f} 秒')  # 打印成功信息
                return True  # 返回启动成功
        except requests.exceptions.ConnectionError:  # 如果连接被拒绝
            pass  # 服务尚未就绪，继续等待
        time.sleep(2)  # 每 2 秒重试一次

    print(f'服务未在 {max_wait} 秒内启动，请检查日志')  # 打印超时警告
    return False  # 返回启动失败


print('通用启动函数 launch_model() 已定义完成')  # 打印函数定义完成提示
print('支持参数: model_path, host, port, trust_remote_code, dtype, extra_args, max_wait')  # 打印支持的参数列表

In [ ]:
# === Demo 1：Qwen2.5 系列 ===
from openai import OpenAI  # 导入 OpenAI 客户端

print('=' * 50)  # 打印分隔线
print('Demo 1：部署 Qwen2.5-0.5B-Instruct')  # 打印 Demo 标题
print('=' * 50)  # 打印分隔线

success = launch_model(  # 调用通用启动函数部署 Qwen2.5
    model_path='Qwen/Qwen2.5-0.5B-Instruct',  # 指定 Qwen2.5-0.5B 模型
    dtype='auto',  # 自动选择数据类型
)  # 启动完成

if success:  # 如果服务启动成功
    client = OpenAI(  # 创建 OpenAI 兼容客户端
        base_url='http://127.0.0.1:30000/v1',  # 指定 SGLang 服务地址
        api_key='EMPTY',  # API 密钥（SGLang 不需要真实密钥）
    )  # 客户端创建完成

    # 测试一：Chat 对话补全模式
    print('\n--- 测试一：Chat 对话模式 ---')  # 打印测试一标题
    chat_resp = client.chat.completions.create(  # 发送对话补全请求
        model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型名称
        messages=[  # 构建消息列表
            {"role": "system", "content": "你是一个专业的中文助手。"},  # 系统提示：设定助手角色
            {"role": "user", "content": "请简要介绍一下杭州西湖的历史和文化价值。"},  # 用户问题：关于西湖
        ],  # 消息列表结束
        max_tokens=200,  # 最大输出 token 数
    )  # 请求完成
    print(f'Chat 回复:\n{chat_resp.choices[0].message.content}')  # 打印对话回复

    # 测试二：Text Completion 文本补全模式
    print('\n--- 测试二：Text Completion 模式 ---')  # 打印测试二标题
    try:  # 尝试使用文本补全模式
        comp_resp = client.completions.create(  # 发送文本补全请求
            model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型名称
            prompt='中国的首都是',  # 输入待补全的提示文本
            max_tokens=100,  # 最大输出 token 数
        )  # 请求完成
        print(f'补全结果:\n中国的首都是{comp_resp.choices[0].text}')  # 打印拼接后的补全结果
    except Exception as e:  # 如果文本补全模式不可用
        print(f'文本补全模式不可用: {e}')  # 打印错误信息
else:  # 如果服务启动失败
    print('Qwen2.5 服务启动失败，请检查环境配置')  # 打印失败提示

In [ ]:
# === Demo 2：DeepSeek 系列 ===
from openai import OpenAI  # 导入 OpenAI 客户端

print('=' * 50)  # 打印分隔线
print('Demo 2：部署 DeepSeek-R1-Distill-Qwen-1.5B')  # 打印 Demo 标题
print('=' * 50)  # 打印分隔线
print('注意：此模型需要 ≥ 8GB 显存\n')  # 提示显存要求

print('启动命令预览：')  # 打印命令预览标题
print('python -m sglang.launch_server \\')  # 打印命令第一行
print('    --model-path deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \\')  # 打印模型路径参数
print('    --host 127.0.0.1 --port 30000 \\')  # 打印地址和端口参数
print('    --dtype auto\n')  # 打印数据类型参数

try:  # 尝试部署 DeepSeek 模型
    success = launch_model(  # 调用通用启动函数
        model_path='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',  # 指定 DeepSeek 蒸馏模型
        dtype='auto',  # 自动选择数据类型
    )  # 启动完成

    if success:  # 如果启动成功
        client = OpenAI(  # 创建 OpenAI 兼容客户端
            base_url='http://127.0.0.1:30000/v1',  # 指定服务地址
            api_key='EMPTY',  # API 密钥
        )  # 客户端创建完成

        print('\n--- 测试：中文推理问题 ---')  # 打印测试标题
        response = client.chat.completions.create(  # 发送推理问题请求
            model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',  # 指定 DeepSeek 模型
            messages=[  # 构建消息列表
                {"role": "user", "content": "小明有 5 个苹果，给了小红 2 个，又买了 3 个，请问小明现在有几个苹果？请一步步推理。"}  # 用户推理问题
            ],  # 消息列表结束
            max_tokens=300,  # 最大输出 token 数
        )  # 请求完成
        print(f'DeepSeek 回复:\n{response.choices[0].message.content}')  # 打印推理回复
    else:  # 如果启动失败
        print('DeepSeek 模型启动失败，可能是显存不足')  # 打印失败原因

except Exception as e:  # 捕获所有异常
    print(f'DeepSeek 部署过程出错: {e}')  # 打印异常信息
    print('可能原因：显存不足（需要 ≥ 8GB）或模型未下载完成')  # 提示可能原因
    print('如果显存不足，请继续使用 Qwen2.5-0.5B 进行后续测试')  # 提示替代方案

## 第三部分：适配要点总结

部署国产开源大模型时，请注意以下关键要点：

- **确认模型在 SGLang 支持列表中：** SGLang 支持大部分 HuggingFace Transformers 架构的模型，但并非所有模型都已适配。建议查阅 [SGLang 官方文档](https://sgl-project.github.io/) 确认支持情况。

- **部分模型需要 `--trust-remote-code`：** 如 ChatGLM 系列等使用自定义模型代码的模型，必须添加此参数才能正常加载。不添加会导致模型初始化失败。

- **`--dtype` 建议使用 `auto`：** 让框架根据模型配置和 GPU 算力自动选择最优数据类型。手动指定 `bfloat16` 在不支持的旧 GPU 上会报错。

- **中文 tokenizer 一般自动加载：** HuggingFace 上的国产模型通常在 `tokenizer_config.json` 中定义好了中文分词器配置，SGLang 会自动识别并加载，无需额外配置。

- **模型下载注意事项：** 首次运行时模型会自动从 HuggingFace 下载。国内用户若下载缓慢，可配置 `HF_ENDPOINT` 环境变量使用镜像站加速。

In [ ]:
# === 中文推理效果测试 ===
from openai import OpenAI  # 导入 OpenAI 客户端

client = OpenAI(  # 创建 OpenAI 兼容客户端
    base_url='http://127.0.0.1:30000/v1',  # 指定当前运行的 SGLang 服务地址
    api_key='EMPTY',  # API 密钥（SGLang 不需要真实密钥）
)  # 客户端创建完成

try:  # 尝试获取当前运行的模型信息
    models_resp = client.models.list()  # 请求模型列表
    current_model = models_resp.data[0].id  # 获取当前加载的模型 ID
    print(f'当前运行的模型: {current_model}\n')  # 打印模型名称
except Exception as e:  # 如果获取模型信息失败
    current_model = 'Qwen/Qwen2.5-0.5B-Instruct'  # 使用默认模型名称
    print(f'获取模型信息失败，使用默认模型名: {current_model}\n')  # 打印提示

tasks = [  # 定义三个中文推理任务
    {  # 任务一：英译中翻译
        'name': '翻译任务',  # 任务名称
        'prompt': "请将以下英文翻译成中文：'Artificial intelligence is transforming the way we live and work, creating new opportunities and challenges.'",  # 翻译提示
    },  # 任务一定义结束
    {  # 任务二：文本摘要
        'name': '摘要任务',  # 任务名称
        'prompt': '请用一句话概括以下内容：人工智能技术近年来发展迅速，在医疗诊断、自动驾驶、智能教育、金融风控等多个领域得到广泛应用。深度学习、自然语言处理、计算机视觉等核心技术不断取得突破，推动了各行各业的智能化转型升级。',  # 摘要提示
    },  # 任务二定义结束
    {  # 任务三：知识问答
        'name': '问答任务',  # 任务名称
        'prompt': '请回答：什么是大语言模型（LLM）？它的主要特点有哪些？请用简洁的中文回答。',  # 问答提示
    },  # 任务三定义结束
]  # 任务列表定义完成

for i, task in enumerate(tasks, 1):  # 遍历任务列表，序号从 1 开始
    print(f'{"=" * 50}')  # 打印分隔线
    print(f'任务 {i}：{task["name"]}')  # 打印当前任务名称
    print(f'{"=" * 50}')  # 打印分隔线
    print(f'输入: {task["prompt"]}\n')  # 打印输入提示文本

    try:  # 尝试发送推理请求
        response = client.chat.completions.create(  # 发送对话补全请求
            model=current_model,  # 使用当前运行的模型
            messages=[  # 构建消息列表
                {"role": "user", "content": task['prompt']}  # 将任务提示作为用户消息
            ],  # 消息列表结束
            max_tokens=200,  # 最大输出 token 数
        )  # 请求完成
        print(f'回复: {response.choices[0].message.content}\n')  # 打印模型回复
    except Exception as e:  # 如果请求失败
        print(f'请求失败: {e}\n')  # 打印错误信息

print('中文推理效果测试完成！')  # 打印所有测试完成提示

In [ ]:
# === 清理资源 ===
import os  # 导入操作系统模块
import signal  # 导入信号模块

try:  # 尝试终止服务进程
    if server_process is not None:  # 如果服务进程变量存在且不为 None
        server_process.terminate()  # 发送 SIGTERM 信号终止服务
        server_process.wait(timeout=10)  # 等待进程退出，最多 10 秒
        print(f'服务进程 (PID={server_process.pid}) 已终止')  # 打印终止成功信息
    else:  # 如果服务进程为 None
        print('没有需要清理的服务进程')  # 打印提示
except NameError:  # 如果 server_process 变量未定义
    print('未找到服务进程变量，可能未启动或已在其他单元格中清理')  # 打印提示
except Exception as e:  # 其他异常
    print(f'终止进程时出错: {e}')  # 打印错误信息
    try:  # 尝试强制终止
        os.kill(server_process.pid, signal.SIGKILL)  # 发送 SIGKILL 强制终止进程
        print('已强制终止进程')  # 打印强制终止成功信息
    except Exception:  # 如果强制终止也失败
        print('请手动运行: kill -9 <PID> 终止进程')  # 提示手动终止

print('资源清理完成')  # 打印清理完成提示

## 常见问题排查

### 问题一：模型不在 SGLang 支持列表中

**错误信息：** `ValueError: Unknown model architecture` 或类似的架构不支持提示

**解决方法：**
- 查阅 [SGLang 官方文档](https://sgl-project.github.io/) 的模型支持列表
- 确认模型的 `config.json` 中 `architectures` 字段是否为 SGLang 已适配的架构
- 如果模型使用非标准架构（如 ChatGLM 的 `ChatGLMForConditionalGeneration`），需确认 SGLang 版本已内置支持
- 可尝试升级 SGLang 到最新版本：`pip install --upgrade sglang`

### 问题二：Tokenizer 加载失败

**错误信息：** `OSError: Can't load tokenizer` 或 `KeyError: 'tokenizer_class'`

**解决方法：**
- 添加 `--trust-remote-code` 参数（部分模型的 tokenizer 依赖自定义代码）
- 检查模型是否完整下载（HuggingFace 下载可能中断）
- 确认 `transformers` 库版本足够新：`pip install --upgrade transformers`
- 国内用户可设置镜像加速：`export HF_ENDPOINT=https://hf-mirror.com`

---

**结语：** SGLang 为国产开源大模型提供了高性能的推理服务支持。通过本 Notebook，你已学会如何使用通用启动函数部署不同的中文模型，并了解了适配过程中的关键注意事项。建议根据自己的硬件条件选择合适的模型，从轻量级的 Qwen2.5-0.5B 开始逐步探索。